In [4]:
# Copyright © 2025 Hao Wu
import torch
from torch import nn
import math
from timm.layers import DropPath, trunc_normal_
import torch.nn.functional as F


def stride_generator(N, reverse=False):
    strides = [1, 2] * 10
    if reverse:
        return list(reversed(strides[:N]))
    else:
        return strides[:N]
    
class MLP(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super(MLP, self).__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

class ConvMLP(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0.):
        super(ConvMLP, self).__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1 = nn.Conv2d(in_features, hidden_features, 1)
        self.act = act_layer()
        self.fc2 = nn.Conv2d(hidden_features, out_features, 1)
        self.drop = nn.Dropout(drop)

    def forward(self, x):
        x = self.fc1(x)
        x = self.act(x)
        x = self.drop(x)
        x = self.fc2(x)
        x = self.drop(x)
        return x

# class Attention(nn.Module):
#     def __init__(self, dim, num_heads=8, qkv_bias=False, qk_scale=None, attn_drop=0., proj_drop=0.):
#         super(Attention, self).__init__()
#         assert dim % num_heads == 0
#         self.num_heads = num_heads
#         self.dim = dim
#         self.head_dim = dim // num_heads
#         self.scale = qk_scale or self.head_dim ** -0.5

#         self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
#         self.proj = nn.Linear(dim, dim)
#         self.attn_drop = float(attn_drop)
#         self.proj_drop = nn.Dropout(proj_drop)

#     def forward(self, x):
#         B, N, C = x.shape
#         h, d = self.num_heads, self.head_dim

#         qkv = self.qkv(x).view(B, N, 3, h, d).transpose(1, 3)
#         q, k, v = qkv[:, :, 0], qkv[:, :, 1], qkv[:, :, 2]

#         q = q.reshape(B, h, N, d).contiguous()
#         k = k.reshape(B, h, N, d).contiguous()
#         v = v.reshape(B, h, N, d).contiguous()

#         p = self.attn_drop if (self.training and self.attn_drop > 0) else 0.0

#         y = F.scaled_dot_product_attention(
#             q, k, v,
#             attn_mask=None,
#             dropout_p=p,
#             is_causal=False
#         )

#         y = y.reshape(B, h, N, d).transpose(1, 2).reshape(B, N, C)
#         y = self.proj(y)
#         y = self.proj_drop(y)
#         return y

class Attention(nn.Module):
    def __init__(self, dim, num_heads=8, qkv_bias=False, qk_scale=None, attn_drop=0., proj_drop=0.):
        super(Attention, self).__init__()
        self.num_heads = num_heads
        head_dim = dim // num_heads
        self.scale = qk_scale or head_dim ** -0.5

        self.qkv = nn.Linear(dim, dim * 3, bias=qkv_bias)
        self.attn_drop = nn.Dropout(attn_drop)
        self.proj = nn.Linear(dim, dim)
        self.proj_drop = nn.Dropout(proj_drop)

    def forward(self, x):
        B, N, C = x.shape
        qkv = (
            self.qkv(x)
            .reshape(B, N, 3, self.num_heads, C // self.num_heads)
            .permute(2, 0, 3, 1, 4)
        )
        q, k, v = qkv[0], qkv[1], qkv[2]

        attn = (q @ k.transpose(-2, -1)) * self.scale
        attn = attn.softmax(dim=-1)
        attn = self.attn_drop(attn)

        x = (attn @ v).transpose(1, 2).reshape(B, N, C)
        x = self.proj(x)
        x = self.proj_drop(x)
        return x


class ConvBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads=4,
        mlp_ratio=4.,
        qkv_bias=False,
        qk_scale=None,
        drop=0.,
        attn_drop=0.,
        drop_path=0.,
        act_layer=nn.GELU,
        norm_layer=nn.LayerNorm
    ):
        super(ConvBlock, self).__init__()
        self.pos_embed = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)
        self.norm1 = nn.BatchNorm2d(dim)
        self.conv1 = nn.Conv2d(dim, dim, 1)
        self.conv2 = nn.Conv2d(dim, dim, 1)
        self.attn = nn.Conv2d(dim, dim, 5, padding=2, groups=dim)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = nn.BatchNorm2d(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = ConvMLP(
            in_features=dim,
            hidden_features=mlp_hidden_dim,
            act_layer=act_layer,
            drop=drop
        )

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, (nn.LayerNorm, nn.GroupNorm, nn.BatchNorm2d)):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)
        elif isinstance(m, nn.Conv2d):
            fan_out = (
                m.kernel_size[0] * m.kernel_size[1] * m.out_channels
            )
            fan_out //= m.groups
            m.weight.data.normal_(0, math.sqrt(2.0 / fan_out))
            if m.bias is not None:
                m.bias.data.zero_()

    @torch.jit.ignore
    def no_weight_decay(self):
        return {}

    def forward(self, x):
        x = x + self.pos_embed(x)
        x = x + self.drop_path(
            self.conv2(self.attn(self.conv1(self.norm1(x))))
        )
        x = x + self.drop_path(self.mlp(self.norm2(x)))
        return x

class SelfAttentionBlock(nn.Module):
    def __init__(
        self,
        dim,
        num_heads,
        mlp_ratio=4.,
        qkv_bias=False,
        qk_scale=None,
        drop=0.,
        attn_drop=0.,
        drop_path=0.,
        init_value=1e-6,
        act_layer=nn.GELU,
        norm_layer=nn.LayerNorm
    ):
        super(SelfAttentionBlock, self).__init__()
        self.pos_embed = nn.Conv2d(dim, dim, 3, padding=1, groups=dim)
        self.norm1 = norm_layer(dim)
        self.attn = Attention(
            dim,
            num_heads=num_heads,
            qkv_bias=qkv_bias,
            qk_scale=qk_scale,
            attn_drop=attn_drop,
            proj_drop=drop
        )
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = norm_layer(dim)
        mlp_hidden_dim = int(dim * mlp_ratio)
        self.mlp = MLP(
            in_features=dim,
            hidden_features=mlp_hidden_dim,
            act_layer=act_layer,
            drop=drop
        )
        self.gamma_1 = nn.Parameter(init_value * torch.ones((dim)), requires_grad=True)
        self.gamma_2 = nn.Parameter(init_value * torch.ones((dim)), requires_grad=True)

        self.apply(self._init_weights)

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if isinstance(m, nn.Linear) and m.bias is not None:
                nn.init.constant_(m.bias, 0)
        elif isinstance(m, (nn.LayerNorm, nn.GroupNorm, nn.BatchNorm2d)):
            nn.init.constant_(m.bias, 0)
            nn.init.constant_(m.weight, 1.0)

    @torch.jit.ignore
    def no_weight_decay(self):
        return {'gamma_1', 'gamma_2'}

    def forward(self, x):
        x = x + self.pos_embed(x)
        B, N, H, W = x.shape
        x = x.flatten(2).transpose(1, 2)
        x = x + self.drop_path(self.gamma_1 * self.attn(self.norm1(x)))
        x = x + self.drop_path(self.gamma_2 * self.mlp(self.norm2(x)))
        x = x.transpose(1, 2).reshape(B, N, H, W)
        return x

def UniformerSubBlock(
    embed_dims,
    mlp_ratio=4.,
    drop=0.,
    drop_path=0.,
    init_value=1e-6,
    block_type='Conv'
):
    assert block_type in ['Conv', 'MHSA']
    if block_type == 'Conv':
        return ConvBlock(dim=embed_dims, mlp_ratio=mlp_ratio, drop=drop, drop_path=drop_path)
    else:
        return SelfAttentionBlock(
            dim=embed_dims,
            num_heads=8,
            mlp_ratio=mlp_ratio,
            qkv_bias=True,
            drop=drop,
            drop_path=drop_path,
            init_value=init_value
        )

class SpatioTemporalEvolutionBlock(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        input_resolution=None,
        mlp_ratio=8.,
        drop=0.0,
        drop_path=0.0,
        layer_i=0,
        force_block_type=None  
    ):
        super(SpatioTemporalEvolutionBlock, self).__init__()
        self.in_channels = in_channels
        self.out_channels = out_channels
        
        if force_block_type is not None:
            block_type = force_block_type
        else:
            block_type = 'MHSA' if in_channels == out_channels and layer_i > 0 else 'Conv'
            
        self.block = UniformerSubBlock(
            in_channels,
            mlp_ratio=mlp_ratio,
            drop=drop,
            drop_path=drop_path,
            block_type=block_type
        )

        if in_channels != out_channels:
            self.reduction = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=1,
                stride=1,
                padding=0
            )

    def forward(self, x):
        z = self.block(x)
        if self.in_channels != self.out_channels:
            z = self.reduction(z)
        return z

class SpatioTemporalEvolution(nn.Module):
    def __init__(
        self,
        channel_in,
        channel_hid,
        N2,
        input_resolution=None,
        mlp_ratio=4.,
        drop=0.0,
        drop_path=0.1,
        force_block_type=None  
    ):
        super(SpatioTemporalEvolution, self).__init__()
        assert N2 >= 2 and mlp_ratio > 1
        self.N2 = N2
        dpr = [x.item() for x in torch.linspace(1e-2, drop_path, self.N2)]

        evolution_layers = [SpatioTemporalEvolutionBlock(
            channel_in,
            channel_hid,
            input_resolution,
            mlp_ratio=mlp_ratio,
            drop=drop,
            drop_path=dpr[0],
            layer_i=0,
            force_block_type=force_block_type
        )]

        for i in range(1, N2 - 1):
            evolution_layers.append(SpatioTemporalEvolutionBlock(
                channel_hid,
                channel_hid,
                input_resolution,
                mlp_ratio=mlp_ratio,
                drop=drop,
                drop_path=dpr[i],
                layer_i=i,
                force_block_type=force_block_type
            ))

        evolution_layers.append(SpatioTemporalEvolutionBlock(
            channel_hid,
            channel_in,
            input_resolution,
            mlp_ratio=mlp_ratio,
            drop=drop,
            drop_path=drop_path,
            layer_i=N2 - 1,
            force_block_type=force_block_type
        ))
        self.enc = nn.Sequential(*evolution_layers)

    def forward(self, x):
        B, T, C, H, W = x.shape
        x = x.reshape(B, T * C, H, W)
        z = x
        for i in range(self.N2):
            z = self.enc[i](z)
        y = z.reshape(B, T, C, H, W)
        return y

class BasicConv2d(nn.Module):
    def __init__(
        self,
        in_channels,
        out_channels,
        kernel_size,
        stride,
        padding,
        transpose=False,
        act_norm=False
    ):
        super(BasicConv2d, self).__init__()
        self.act_norm = act_norm
        if not transpose:
            self.conv = nn.Conv2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding
            )
        else:
            self.conv = nn.ConvTranspose2d(
                in_channels,
                out_channels,
                kernel_size=kernel_size,
                stride=stride,
                padding=padding,
                output_padding=stride // 2
            )
        self.norm = nn.GroupNorm(2, out_channels)
        self.act = nn.LeakyReLU(0.2, inplace=True)

    def forward(self, x):
        y = self.conv(x)
        if self.act_norm:
            y = self.act(self.norm(y))
        return y

class ConvDynamicsLayer(nn.Module):
    def __init__(self, C_in, C_out, stride, transpose=False, act_norm=True):
        super(ConvDynamicsLayer, self).__init__()
        if stride == 1:
            transpose = False
        self.conv = BasicConv2d(
            C_in,
            C_out,
            kernel_size=3,
            stride=stride,
            padding=1,
            transpose=transpose,
            act_norm=act_norm
        )

    def forward(self, x):
        y = self.conv(x)
        return y

class AtmosphericEncoder(nn.Module):
    def __init__(self, C_in, spatial_hidden_dim, num_spatial_layers):
        super(AtmosphericEncoder, self).__init__()
        strides = stride_generator(num_spatial_layers)
        self.enc = nn.Sequential(
            ConvDynamicsLayer(C_in, spatial_hidden_dim, stride=strides[0]),
            *[ConvDynamicsLayer(spatial_hidden_dim, spatial_hidden_dim, stride=s) for s in strides[1:]]
        )

    def forward(self, x):
        enc1 = self.enc[0](x)
        latent = enc1
        for i in range(1, len(self.enc)):
            latent = self.enc[i](latent)
        return latent, enc1

class AtmosphericDecoder(nn.Module):
    def __init__(self, spatial_hidden_dim, C_out, num_spatial_layers):
        super(AtmosphericDecoder, self).__init__()
        strides = stride_generator(num_spatial_layers, reverse=True)
        self.dec = nn.Sequential(
            *[ConvDynamicsLayer(spatial_hidden_dim, spatial_hidden_dim, stride=s, transpose=True) for s in strides[:-1]],
            ConvDynamicsLayer(2 * spatial_hidden_dim, spatial_hidden_dim, stride=strides[-1], transpose=True)
        )
        self.readout = nn.Conv2d(spatial_hidden_dim, C_out, 1)

    def forward(self, hid, enc1=None):
        for i in range(0, len(self.dec) - 1):
            hid = self.dec[i](hid)
        Y = self.dec[-1](torch.cat([hid, enc1], dim=1))
        Y = self.readout(Y)
        return Y

class AtmosphericDecoderNoSkip(nn.Module):
    def __init__(self, spatial_hidden_dim, C_out, num_spatial_layers):
        super(AtmosphericDecoderNoSkip, self).__init__()
        strides = stride_generator(num_spatial_layers, reverse=True)
        self.dec = nn.Sequential(
            *[ConvDynamicsLayer(spatial_hidden_dim, spatial_hidden_dim, stride=s, transpose=True) for s in strides]
        )
        self.readout = nn.Conv2d(spatial_hidden_dim, C_out, 1)

    def forward(self, hid, enc1=None):
        for i in range(len(self.dec)):
            hid = self.dec[i](hid)
        Y = self.readout(hid)
        return Y


class TritonCast(nn.Module):
    def __init__(
        self,
        input_channel,
        spatial_hidden_dim=64,
        output_channels=4,
        temporal_hidden_dim=128,
        num_spatial_layers=4,
        num_temporal_layers=8,
        in_time_seq_length=10,
        out_time_seq_length=10
    ):
        super(TritonCast, self).__init__()
        T = 1
        C = input_channel
        self.output_dim = output_channels
        self.input_time_seq_length = in_time_seq_length
        self.output_time_seq_length = out_time_seq_length
        
        self.atmospheric_encoder = AtmosphericEncoder(C, spatial_hidden_dim, num_spatial_layers)
        self.temporal_evolution = SpatioTemporalEvolution(
            T * spatial_hidden_dim,
            temporal_hidden_dim,
            num_temporal_layers,
            input_resolution=None,
            mlp_ratio=4.0,
            drop_path=0.1
        )
        self.atmospheric_decoder = AtmosphericDecoder(spatial_hidden_dim, self.output_dim, num_spatial_layers)

    def forward(self, input_state):
        batch_size, temporal_length, channels, height, width = input_state.shape
        reshaped_input = input_state.view(batch_size * temporal_length, channels, height, width)
        
        encoded_features, skip_connection = self.atmospheric_encoder(reshaped_input)
        _, encoded_channels, encoded_height, encoded_width = encoded_features.shape
        encoded_features = encoded_features.view(batch_size, temporal_length, encoded_channels, encoded_height, encoded_width)
        
        temporal_bias = encoded_features
        temporal_hidden = self.temporal_evolution(temporal_bias)
        reshaped_hidden = temporal_hidden.view(batch_size * temporal_length, encoded_channels, encoded_height, encoded_width)

        decoded_output = self.atmospheric_decoder(reshaped_hidden, skip_connection)
        final_output = decoded_output.view(batch_size, temporal_length, -1, height, width)
        
        return final_output
class SimpleEncoder(nn.Module):
    def __init__(self, C_in, C_hid, num_layers):
        super(SimpleEncoder, self).__init__()
        layers = []
        in_channels = C_in
        for i in range(num_layers):
            out_channels = C_hid
            stride = 2 if i < num_layers - 1 else 1  # Last layer has stride 1
            layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=stride, padding=1))
            layers.append(nn.GroupNorm(2, out_channels))
            layers.append(nn.LeakyReLU(0.2, inplace=True))
            in_channels = out_channels
        
        self.net = nn.Sequential(*layers)
    
    def forward(self, x):
        return self.net(x), None  # Return None for skip_connection to maintain interface compatibility

class SimpleDecoder(nn.Module):
    def __init__(self, C_hid, C_out, num_layers):
        super(SimpleDecoder, self).__init__()
        layers = []
        in_channels = C_hid
        for i in range(num_layers):
            out_channels = C_hid if i < num_layers - 1 else C_out
            if i > 0:  # First layer has stride 1, others have stride 2
                layers.append(nn.ConvTranspose2d(in_channels, out_channels, kernel_size=3, stride=2, padding=1, output_padding=1))
            else:
                layers.append(nn.Conv2d(in_channels, out_channels, kernel_size=3, stride=1, padding=1))
            
            if i < num_layers - 1:
                layers.append(nn.GroupNorm(2, out_channels))
                layers.append(nn.LeakyReLU(0.2, inplace=True))
            in_channels = out_channels
        
        self.net = nn.Sequential(*layers)
    
    def forward(self, x, skip_connection=None):
        # Skip connection is not used here but kept for interface compatibility
        return self.net(x)
        
class TritonCastFlat(nn.Module):
    def __init__(
        self,
        input_channel,
        spatial_hidden_dim=64,
        output_channels=4,
        temporal_hidden_dim=128,
        num_spatial_layers=4,
        num_temporal_layers=8,
        in_time_seq_length=10,
        out_time_seq_length=10
    ):
        super(TritonCastFlat, self).__init__()
        T = 1
        C = input_channel
        self.output_dim = output_channels
        
        self.flat_encoder = SimpleEncoder(C, spatial_hidden_dim, num_spatial_layers)
        self.temporal_evolution = SpatioTemporalEvolution(
            T * spatial_hidden_dim,
            temporal_hidden_dim,
            num_temporal_layers,
            mlp_ratio=4.0,
            drop_path=0.1
        )
        
        self.flat_decoder = SimpleDecoder(spatial_hidden_dim, self.output_dim, num_spatial_layers)
        
    def forward(self, input_state):
        batch_size, temporal_length, channels, height, width = input_state.shape
        reshaped_input = input_state.view(batch_size * temporal_length, channels, height, width)
        
        encoded_features, _ = self.flat_encoder(reshaped_input)
        _, encoded_channels, encoded_height, encoded_width = encoded_features.shape
        encoded_features = encoded_features.view(batch_size, temporal_length, encoded_channels, encoded_height, encoded_width)
        
        temporal_bias = encoded_features
        temporal_hidden = self.temporal_evolution(temporal_bias)
        reshaped_hidden = temporal_hidden.view(batch_size * temporal_length, encoded_channels, encoded_height, encoded_width)

        decoded_output = self.flat_decoder(reshaped_hidden, None)
        decoded_output = F.interpolate(decoded_output, size=(128, 128), mode='bilinear', align_corners=False)
        final_output = decoded_output.view(batch_size, temporal_length, -1, height, width)

        return final_output


class TritonCastNoSkip(nn.Module):
    def __init__(
        self,
        input_channel,
        spatial_hidden_dim=64,
        output_channels=4,
        temporal_hidden_dim=128,
        num_spatial_layers=4,
        num_temporal_layers=8,
        in_time_seq_length=10,
        out_time_seq_length=10
    ):
        super(TritonCastNoSkip, self).__init__()
        T = 1
        C = input_channel
        self.output_dim = output_channels
        
        self.atmospheric_encoder = AtmosphericEncoder(C, spatial_hidden_dim, num_spatial_layers)
        self.temporal_evolution = SpatioTemporalEvolution(
            T * spatial_hidden_dim,
            temporal_hidden_dim,
            num_temporal_layers,
            mlp_ratio=4.0,
            drop_path=0.1
        )
        self.atmospheric_decoder = AtmosphericDecoderNoSkip(spatial_hidden_dim, self.output_dim, num_spatial_layers)

    def forward(self, input_state):
        batch_size, temporal_length, channels, height, width = input_state.shape
        reshaped_input = input_state.view(batch_size * temporal_length, channels, height, width)
        
        encoded_features, _ = self.atmospheric_encoder(reshaped_input)  # 忽略skip_connection
        _, encoded_channels, encoded_height, encoded_width = encoded_features.shape
        encoded_features = encoded_features.view(batch_size, temporal_length, encoded_channels, encoded_height, encoded_width)
        
        temporal_hidden = self.temporal_evolution(encoded_features)
        reshaped_hidden = temporal_hidden.view(batch_size * temporal_length, encoded_channels, encoded_height, encoded_width)

        decoded_output = self.atmospheric_decoder(reshaped_hidden, None)
        final_output = decoded_output.view(batch_size, temporal_length, -1, height, width)
        
        return final_output

class TritonCastLatentConv(nn.Module):
    def __init__(
        self,
        input_channel,
        spatial_hidden_dim=64,
        output_channels=4,
        temporal_hidden_dim=128,
        num_spatial_layers=4,
        num_temporal_layers=8,
        in_time_seq_length=10,
        out_time_seq_length=10
    ):
        super(TritonCastLatentConv, self).__init__()
        T = 1
        C = input_channel
        self.output_dim = output_channels
        
        self.atmospheric_encoder = AtmosphericEncoder(C, spatial_hidden_dim, num_spatial_layers)
        self.temporal_evolution = SpatioTemporalEvolution(
            T * spatial_hidden_dim,
            temporal_hidden_dim,
            num_temporal_layers,
            mlp_ratio=4.0,
            drop_path=0.1,
            force_block_type='Conv'
        )
        self.atmospheric_decoder = AtmosphericDecoder(spatial_hidden_dim, self.output_dim, num_spatial_layers)

    def forward(self, input_state):
        batch_size, temporal_length, channels, height, width = input_state.shape
        reshaped_input = input_state.view(batch_size * temporal_length, channels, height, width)
        
        encoded_features, skip_connection = self.atmospheric_encoder(reshaped_input)
        _, encoded_channels, encoded_height, encoded_width = encoded_features.shape
        encoded_features = encoded_features.view(batch_size, temporal_length, encoded_channels, encoded_height, encoded_width)
        # print("encoded_features shape", encoded_features.shape)
        # temporal_hidden = self.temporal_evolution(encoded_features)
        temporal_hidden = encoded_features # No LDC
        # print("temporal_hidden shape", temporal_hidden.shape)
        reshaped_hidden = temporal_hidden.view(batch_size * temporal_length, encoded_channels, encoded_height, encoded_width)

        decoded_output = self.atmospheric_decoder(reshaped_hidden, skip_connection)
        final_output = decoded_output.view(batch_size, temporal_length, -1, height, width)
        
        return final_output

class TritonCastNet(nn.Module):
    def __init__(
        self,
        input_channel,
        spatial_hidden_dim=64,
        output_channels=4,
        temporal_hidden_dim=128,
        num_spatial_layers=4,
        num_temporal_layers=8,
        in_time_seq_length=10,
        out_time_seq_length=10
    ):
        super(TritonCastNet, self).__init__()
        T = 1
        C = input_channel
        self.output_dim = output_channels
        
        self.atmospheric_encoder = AtmosphericEncoder(C, spatial_hidden_dim, num_spatial_layers)
        self.temporal_evolution = SpatioTemporalEvolution(
            T * spatial_hidden_dim,
            temporal_hidden_dim,
            num_temporal_layers,
            mlp_ratio=4.0,
            drop_path=0.1,
            force_block_type='MHSA'
        )
        self.atmospheric_decoder = AtmosphericDecoder(spatial_hidden_dim, self.output_dim, num_spatial_layers)

    def forward(self, input_state):
        batch_size, temporal_length, channels, height, width = input_state.shape
        reshaped_input = input_state.view(batch_size * temporal_length, channels, height, width)
        
        encoded_features, skip_connection = self.atmospheric_encoder(reshaped_input)
        _, encoded_channels, encoded_height, encoded_width = encoded_features.shape
        encoded_features = encoded_features.view(batch_size, temporal_length, encoded_channels, encoded_height, encoded_width)
        
        temporal_hidden = self.temporal_evolution(encoded_features)
        reshaped_hidden = temporal_hidden.view(batch_size * temporal_length, encoded_channels, encoded_height, encoded_width)

        decoded_output = self.atmospheric_decoder(reshaped_hidden, skip_connection)
        final_output = decoded_output.view(batch_size, temporal_length, -1, height, width)
        
        return final_output


def create_model(model_type, **kwargs):
    model_classes = {
        'full': TritonCastNet,
        'flat': TritonCastFlat,
        'no_skip': TritonCastNoSkip,
        'no_ldc': TritonCastLatentConv,
    }
    
    if model_type not in model_classes:
        raise ValueError(f"Unknown model type: {model_type}. Available: {list(model_classes.keys())}")
    
    return model_classes[model_type](**kwargs)



def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

# ====================Test ====================

if __name__ == '__main__':
    input_shape = (1, 10, 2, 720, 1440)  # (batch, time, channels, height, width)
    inputs = torch.randn(input_shape)
    
    model_config = {
        'input_channel': 2,
        'spatial_hidden_dim': 64,
        'output_channels': 2,
        'temporal_hidden_dim': 256,
        'num_spatial_layers': 4,
        'num_temporal_layers': 4
    }
    
    # model_types = ['full',  'flat', 'no_skip', 'no_ldc']
    model_types = ['no_ldc']
    print("Model Test Results:")
    print("=" * 80)
    
    for model_type in model_types:
        model = create_model(model_type, **model_config)
        output = model(inputs)
        params = count_parameters(model)
        
        print(f"{model_type:12s}: Input shape {tuple(output)}, Param {params/1e9:.3f}B")
    
    print("=" * 80)

Model Test Results:
no_ldc      : Input shape (tensor([[[[ 2.4061e-01, -5.3863e-01, -2.4136e-01,  ..., -1.8155e-01,
           -4.2065e-02, -1.1486e-01],
          [-9.3335e-02,  2.6246e-01, -4.3141e-02,  ..., -6.7470e-01,
            5.1959e-02, -1.3890e-01],
          [ 7.2205e-02,  1.7369e-01, -1.1539e-01,  ..., -2.3612e-01,
           -2.5599e-01,  6.3614e-02],
          ...,
          [-9.5763e-02,  5.2142e-02, -2.6317e-01,  ...,  2.0476e-01,
           -3.3547e-01, -1.4129e-01],
          [-2.2478e-01, -4.9548e-02, -5.2456e-01,  ..., -6.7316e-01,
           -1.2995e-01, -1.2819e-02],
          [ 8.6417e-02,  1.1011e-01, -9.9528e-02,  ...,  5.7871e-02,
           -2.4186e-02, -8.0670e-02]],

         [[-2.7978e-01, -1.1518e-01, -4.1435e-01,  ..., -3.3468e-01,
           -5.5744e-02, -1.0125e-01],
          [-1.3739e-01, -3.5740e-01,  2.1436e-01,  ...,  2.2092e-02,
            9.2040e-02, -7.8202e-03],
          [ 1.6816e-01,  1.6160e-02,  5.6362e-02,  ...,  1.1678e-01,
           